# Huấn luyện DermaSense AI trên Kaggle (Khuyên dùng)
Kaggle cung cấp 2 chiếc Card GPU T4 miễn phí. Tốc độ sẽ nhanh gấp đôi Colab.

## Bước 1: Bật GPU
Bên cột bên phải (Settings) -> Kéo xuống phần **ACCELERATOR** -> Chọn **GPU T4 x2**.

## Bước 2: Clone Code từ GitHub

In [ ]:
import os

REPO_URL = 'https://github.com/PHANQUOCTHANG/DermaSense_AI.git'
PROJECT_DIR = '/kaggle/working/DermaSense_AI'

if os.path.exists(PROJECT_DIR):
    %cd {PROJECT_DIR}
    !git pull origin main
    print('Da cap nhat code moi nhat tu GitHub!')
else:
    !git clone {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}
    print('Da clone thanh cong tu GitHub!')

## Bước 3: Tải Dữ Liệu Tự Động (DermNet, HAM10000, PAD-UFES)
Chạy Cell này để tự động tải và trộn 3 tập dữ liệu vàng.

In [ ]:
import os

# Dung API Token kieu moi cua Kaggle (chi can 1 dong duy nhat)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_54345836fa9c02e730cbf81201dbbdf0"

!pip install -q kaggle
print("\nDang tai du lieu DermNet (1.7GB)...")
!kaggle datasets download -q -d shubhamgoel27/dermnet -p data/raw/dermnet_raw --unzip

print("\nDang tai du lieu HAM10000 (3GB)...")
!kaggle datasets download -q -d kmader/skin-cancer-mnist-ham10000 -p data/raw/ham10000_raw --unzip

print("\nDang tai du lieu PAD-UFES-20 (1GB)...")
!kaggle datasets download -q -d wanderdust/pad-ufes-20 -p data/raw/pad_ufes_raw --unzip

print("\nTai xong tat ca 3 tap du lieu!")

## Bước 4: Xử lý và Hợp nhất Dữ Liệu

In [ ]:
import shutil, random, os
import pandas as pd
import numpy as np
from pathlib import Path

img_out = Path('data/raw/images')
if img_out.exists(): shutil.rmtree(img_out)
img_out.mkdir(parents=True, exist_ok=True)

metadata_list = []

print("--- XU LY DERMNET ---")
dermnet_dir = Path('data/raw/dermnet_raw')
d_count = 0
if dermnet_dir.exists():
    for split_dir in ['train', 'test']:
        split_path = dermnet_dir / split_dir
        if split_path.exists():
            for cls_dir in split_path.iterdir():
                if cls_dir.is_dir():
                    for img_path in cls_dir.glob('*.*'):
                        if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                            img_id = f'DN_{d_count:07d}'
                            new_path = img_out / f'{img_id}{img_path.suffix}'
                            shutil.copy2(img_path, new_path)
                            metadata_list.append({
                                'image_id': img_id, 'diagnosis': cls_dir.name, 'age': 45, 'sex': None, 
                                'anatom_site': None, 'duration': 30, 'symptoms': 'none', 'skin_type': 'III', 'family_history': 'no'
                            })
                            d_count += 1
print(f"Da xu ly {d_count} anh DermNet.")

print("--- XU LY HAM10000 ---")
ham_dir = Path('data/raw/ham10000_raw')
h_count = 0
if ham_dir.exists():
    ham_map = {'nv': 'Melanoma and Nevi', 'mel': 'Melanoma and Nevi', 'bkl': 'Seborrheic Keratoses and Benign Tumors', 'bcc': 'Actinic Keratosis and Skin Cancer', 'akiec': 'Actinic Keratosis and Skin Cancer', 'vasc': 'Vascular Tumors', 'df': 'Seborrheic Keratoses and Benign Tumors'}
    ham_csv = ham_dir / 'HAM10000_metadata.csv'
    if ham_csv.exists():
        df_ham = pd.read_csv(ham_csv)
        for _, row in df_ham.iterrows():
            img_id_orig = row['image_id']
            img_path = None
            for part in ['HAM10000_images_part_1', 'HAM10000_images_part_2']:
                p = ham_dir / part / f"{img_id_orig}.jpg"
                if p.exists(): img_path = p; break
            if img_path:
                img_id = f'HAM_{h_count:07d}'
                new_path = img_out / f'{img_id}.jpg'
                shutil.copy2(img_path, new_path)
                label = ham_map.get(row['dx'], 'Systemic Diseases')
                age = row['age'] if pd.notna(row['age']) else 45
                sex = row['sex'] if pd.notna(row['sex']) and row['sex'] in ['male', 'female'] else None
                loc = row['localization'] if pd.notna(row['localization']) and row['localization'] != 'unknown' else None
                metadata_list.append({'image_id': img_id, 'diagnosis': label, 'age': age, 'sex': sex, 'anatom_site': loc, 'duration': 30, 'symptoms': 'none', 'skin_type': 'III', 'family_history': 'no'})
                h_count += 1
print(f"Da xu ly {h_count} anh HAM10000.")

print("--- XU LY PAD-UFES-20 ---")
pad_dir = Path('data/raw/pad_ufes_raw')
p_count = 0
if pad_dir.exists():
    pad_map = {'BCC': 'Actinic Keratosis and Skin Cancer', 'SCC': 'Actinic Keratosis and Skin Cancer', 'ACK': 'Actinic Keratosis and Skin Cancer', 'SEK': 'Seborrheic Keratoses and Benign Tumors', 'BOD': 'Seborrheic Keratoses and Benign Tumors', 'MEL': 'Melanoma and Nevi', 'NEV': 'Melanoma and Nevi'}
    pad_csv = pad_dir / 'pad_ufes_20.csv'
    if pad_csv.exists():
        df_pad = pd.read_csv(pad_csv)
        for _, row in df_pad.iterrows():
            img_id_orig = row['img_id']
            img_path = None
            for folder in pad_dir.iterdir():
                if folder.is_dir() and folder.name.startswith('images_'):
                    p = folder / img_id_orig
                    if p.exists(): img_path = p; break
            if img_path:
                img_id = f'PAD_{p_count:07d}'
                new_path = img_out / f'{img_id}{img_path.suffix}'
                shutil.copy2(img_path, new_path)
                label = pad_map.get(row['diagnostic'], 'Systemic Diseases')
                age = row['age'] if pd.notna(row['age']) else 45
                sex = str(row['gender']).lower() if pd.notna(row['gender']) else None
                if sex not in ['male', 'female']: sex = None
                skin_map = {'1.0': 'I', '2.0': 'II', '3.0': 'III', '4.0': 'IV', '5.0': 'V', '6.0': 'VI'}
                skin_type = skin_map.get(str(row['fitzpatrick']), 'III')
                symptom = 'none'
                if row.get('itch') == 'True' or row.get('itch') == True: symptom = 'itch'
                elif row.get('bleed') == 'True' or row.get('bleed') == True: symptom = 'bleeding'
                elif row.get('hurt') == 'True' or row.get('hurt') == True: symptom = 'pain'
                metadata_list.append({'image_id': img_id, 'diagnosis': label, 'age': age, 'sex': sex, 'anatom_site': None, 'duration': 30, 'symptoms': symptom, 'skin_type': skin_type, 'family_history': 'no'})
                p_count += 1
print(f"Da xu ly {p_count} anh PAD-UFES-20.")

print("--- GHI FILE METADATA.CSV CHUNG ---")
df_all = pd.DataFrame(metadata_list)
df_all.to_csv('data/raw/metadata.csv', index=False)
print(f"Hoan tat! Tong cong {len(df_all)} anh + metadata.csv")

## Bước 5: Cài Thư viện & Thiết lập Output Kaggle

In [ ]:
!pip install -q timm albumentations opencv-python-headless pyyaml
import yaml, os
from pathlib import Path

# KAGGLE KHONG DUNG GOOGLE DRIVE! Luu truc tiep vao /kaggle/working/
kaggle_ckpt = '/kaggle/working/DermaSense_AI/models/checkpoints'
os.makedirs(kaggle_ckpt, exist_ok=True)

with open('configs/stage3_train_multimodal.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
cfg['training']['checkpoint_dir'] = kaggle_ckpt
with open('configs/stage3_train_multimodal.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f)

last_ckpt = os.path.join(kaggle_ckpt, 'last_checkpoint.pt')
if os.path.exists(last_ckpt):
    print('Tim thay checkpoint cu! AI se hoc tiep tu cho cu.')
else:
    print('Chua co checkpoint cu. Bat dau huan luyen tu dau.')

print(f'Checkpoint se duoc luu tai: {kaggle_ckpt}')

## Bước 6: HUẤN LUYỆN ĐA PHƯƠNG THỨC

In [ ]:
!PYTHONIOENCODING="utf-8" python -m src.pipelines.run_stage3_train_multimodal

## Bước 7: Tải File Trọng Số Về Máy
Cột bên phải (Output) -> Tải file `best_model.pt` về máy của bạn.

In [ ]:
import shutil, os

src_stage = "/kaggle/working/DermaSense_AI/models/checkpoints/stage_b/best_model.pt"
src_best = "/kaggle/working/DermaSense_AI/models/checkpoints/best_model.pt"
dst = "/kaggle/working/best_model.pt"

if os.path.exists(src_stage):
    shutil.copy2(src_stage, dst)
    size_mb = os.path.getsize(dst) / 1024 / 1024
    print(f"Da copy best_model.pt ra Output! Dung luong: {size_mb:.1f} MB")
elif os.path.exists(src_best):
    shutil.copy2(src_best, dst)
    size_mb = os.path.getsize(dst) / 1024 / 1024
    print(f"Da copy best_model.pt ra Output! Dung luong: {size_mb:.1f} MB")
else:
    print("CANH BAO: Khong tim thay file best_model.pt!")